# XGBoost PU Bagging — 关键波段输入（CN3839 / CN4142 / CH4300）

**实验目的：** 与全波段基线 `ML_XGB_PU_threshold` 对比，只把三个 CN/CH 分子带窗口作为模型输入，检验"仅凭关键波段"能否达到（或接近）全波段 700 维的性能，进而判断 CN 判别信息是否主要集中在这三个分子带上。

**核心改动（相对基线）：**
1. 输入从全波段 **700 维** → 三个关键分子带窗口拼接，共 **182 维**
2. 分子带窗口：CN3839 (3830–3883Å)、CN4142 (4120–4216Å)、CH4300 (4285–4315Å)
3. 其余流程（PU Bagging T=500、已知 CN 星标定阈值、候选体导出）与基线完全一致，保证可比性

**预期：** 若关键波段性能与全波段相当，说明分子带承载了绝大部分判别信息；若明显下降，说明连续谱形态（温度/金属丰度背景）也提供了重要线索。

In [ ]:
# 共享数据加载与基础库导入

import sys, os, time
from pathlib import Path

# 智能定位项目根目录：向上查找直到同时存在 ML/ 与 Data/
_PROJECT_ROOT = Path(os.getcwd())
for _ in range(5):
    if (_PROJECT_ROOT / "ML").exists() and (_PROJECT_ROOT / "Data").exists():
        break
    _PROJECT_ROOT = _PROJECT_ROOT.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 10})
plt.rcParams.update({'axes.labelsize': 'large'})
import warnings
warnings.filterwarnings('ignore')

from PhaseSummary.shared.data_loader import BAND_DEFS, MOLECULAR_BAND_RANGES, CN_BAND_NAMES
from build_dr13_all_cache import load_dr13_all_cache

t0 = time.time()
data = load_dr13_all_cache()
X_clean = data['X_clean']
stars_clean = data['stars_clean']
feature_df = data['feature_df']
common_wave = data['common_wave']

print(f"数据加载完成 ({time.time()-t0:.0f}s):")
print(f"  归一化光谱 X_clean: {X_clean.shape}")
print(f"  恒星数量:           {len(stars_clean)}")
print(f"  已知CN星:          {(stars_clean['label']==1).sum()}")
print(f"  波长范围:           {common_wave[0]:.0f}-{common_wave[-1]:.0f} Å")


## 1. 关键波段提取

从连续谱归一化光谱中裁剪出三个 CN/CH 分子带窗口并拼接。分子带是 CN 吸收最强、最具有判别力的区域；这里**仅输入分子带本身**（不含两侧连续谱），考察"关键波段"是否足够支撑识别。

In [ ]:
# 提取三个关键分子带波段（CN3839 / CN4142 / CH4300），拼接成 (N, 182) 输入
# MOLECULAR_BAND_RANGES = [(3830, 3883), (4120, 4216), (4285, 4315)] —— 只取分子带窗口

band_ranges = MOLECULAR_BAND_RANGES  # 三个关键波段
band_masks = [(common_wave >= l1) & (common_wave <= l2) for (l1, l2) in band_ranges]

X_bands = np.concatenate([X_clean[:, m] for m in band_masks], axis=1).astype(np.float32)
band_wave = np.concatenate([common_wave[m] for m in band_masks])
n_px_per_band = [int(m.sum()) for m in band_masks]

print("关键波段输入:")
for (l1, l2), npx in zip(band_ranges, n_px_per_band):
    print(f"  {l1}-{l2} Å : {npx} 像素")
print(f"  总输入维度: {X_bands.shape[1]}  (全波段为 {X_clean.shape[1]})")
print(f"  维度压缩: {X_clean.shape[1]} -> {X_bands.shape[1]}  ({X_bands.shape[1]/X_clean.shape[1]*100:.1f}%)")


## 2. 标准化与数据划分

与基线完全一致的标准化 + 分层划分（seed=42, test=15%, val=18%），保证两者在完全相同的数据划分上比较。

In [ ]:
# 标准化 + 分层划分（与基线一致）

import time, random
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

random_seed = 42

y_all = stars_clean['label'].map({1: 1, -1: 0}).values.astype(int)
X_spectra = X_bands.astype(np.float32)   # 关键波段输入 (N, 182)

ss = StandardScaler()
X_spec_scaled = ss.fit_transform(X_spectra).astype(np.float32)

all_idx = np.arange(len(y_all))
tv_idx, test_idx = train_test_split(
    all_idx, test_size=0.15, stratify=y_all, random_state=random_seed)
tr_idx, val_idx = train_test_split(
    tv_idx, test_size=0.18, stratify=y_all[tv_idx], random_state=random_seed)

n_pos = int(y_all.sum())
print(f"总样本: {len(y_all):,}  正样本(P): {n_pos}  负/未标注(U): {len(y_all)-n_pos:,}")
print(f"训练集: {len(tr_idx):,}  测试集: {len(test_idx):,}  输入维度: {X_spec_scaled.shape[1]}")


## 3. 运行 PU Bagging（T=500）

在 182 维关键波段输入上运行 PU Bagging，流程与基线完全相同。

In [ ]:
# 运行 PU Bagging (T=500) — 关键波段输入

T = 500
print(f"运行 PU Bagging (T={T}) — 输入维度 {X_spec_scaled.shape[1]}...")

X_tr = X_spec_scaled[tr_idx]
X_te = X_spec_scaled[test_idx]
y_tr = y_all[tr_idx]
y_te = y_all[test_idx]

pos_tr_idx = np.where(y_tr == 1)[0]
n_pos_tr = len(pos_tr_idx)
unl_tr_idx = np.where(y_tr == 0)[0]
X_pos = X_tr[pos_tr_idx]

rng = random.Random(random_seed)

te_prob_sum = np.zeros(len(test_idx), dtype=np.float64)
te_prob_sq = np.zeros(len(test_idx), dtype=np.float64)
all_prob_sum = np.zeros(len(y_all), dtype=np.float64)
all_prob_sq = np.zeros(len(y_all), dtype=np.float64)

xgb_params = {
    "max_depth": 3, "learning_rate": 0.1,
    "subsample": 0.8, "colsample_bytree": 0.8,
    "min_child_weight": 1, "gamma": 0.0,
    "reg_alpha": 0.0, "reg_lambda": 1.0,
    "seed": random_seed, "verbosity": 0, "n_jobs": 1,
}

t0 = time.time()
for t in range(1, T + 1):
    neg_sample = rng.sample(list(unl_tr_idx), n_pos_tr)
    X_neg = X_tr[neg_sample]
    X_bal = np.vstack([X_pos, X_neg])
    y_bal = np.hstack([np.ones(n_pos_tr), np.zeros(n_pos_tr)])

    dtrain = xgb.DMatrix(X_bal, label=y_bal)
    model = xgb.train(xgb_params, dtrain, num_boost_round=50, verbose_eval=False)

    p_te = model.predict(xgb.DMatrix(X_te))
    p_all = model.predict(xgb.DMatrix(X_spec_scaled))

    te_prob_sum += p_te
    te_prob_sq += p_te ** 2
    all_prob_sum += p_all
    all_prob_sq += p_all ** 2

    if t % 100 == 0:
        p_m = te_prob_sum / t
        pr = average_precision_score(y_te, p_m)
        roc = roc_auc_score(y_te, p_m)
        print(f"  [{t:4d}/{T}]  ROC={roc:.4f}  PR={pr:.4f}  ({time.time()-t0:.0f}s)")

p_te_mean = te_prob_sum / T
p_all_mean = all_prob_sum / T
p_all_std = np.sqrt(np.maximum(all_prob_sq / T - p_all_mean**2, 0))

elapsed = time.time() - t0
roc = roc_auc_score(y_te, p_te_mean)
pr = average_precision_score(y_te, p_te_mean)

order = np.argsort(p_te_mean)[::-1]
p50 = y_te[order[:min(50, len(order))]].mean()
p100 = y_te[order[:min(100, len(order))]].mean()

print(f"PU Bagging 完成 ({elapsed:.0f}s)")
print(f"  测试集 ROC-AUC: {roc:.4f}")
print(f"  测试集 PR-AUC: {pr:.4f}")
print(f"  Precision@50: {p50:.4f}")
print(f"  Precision@100: {p100:.4f}")
print(f"  平均概率: {p_all_mean.mean():.4f} ± {p_all_mean.std():.4f}")

# 保存结果
stars_clean['xgb_pu_prob'] = p_all_mean
stars_clean['xgb_pu_std'] = p_all_std


## 4. 性能曲线（PR / ROC）

In [ ]:
# 绘制 PR 曲线与 ROC 曲线
from sklearn.metrics import precision_recall_curve, roc_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

precision, recall, _ = precision_recall_curve(y_te, p_te_mean)
pr_auc = average_precision_score(y_te, p_te_mean)
baseline = y_te.sum() / len(y_te)

ax1 = axes[0]
ax1.plot(recall, precision, 'b-', linewidth=2, label=f'XGB PU key-bands (AP={pr_auc:.3f})')
ax1.axhline(baseline, color='gray', linestyle='--', linewidth=1.0,
            label=f'Random baseline ({baseline:.3f})')
ax1.fill_between(recall, precision, baseline, alpha=0.08, color='blue')
ax1.set_xlabel('Recall'); ax1.set_ylabel('Precision')
ax1.set_title('Precision-Recall Curve')
ax1.legend(fontsize=9, loc='upper right')
ax1.grid(alpha=0.2); ax1.set_xlim(0, 1.02); ax1.set_ylim(0, 1.02)

fpr, tpr, _ = roc_curve(y_te, p_te_mean)
roc_auc = auc(fpr, tpr)
ax2 = axes[1]
ax2.plot(fpr, tpr, 'darkred', linewidth=2, label=f'XGB PU key-bands (AUC={roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], 'gray', linestyle='--', linewidth=1.0, label='Random (AUC=0.500)')
ax2.fill_between(fpr, tpr, 0, alpha=0.08, color='darkred')
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve')
ax2.legend(fontsize=9, loc='lower right')
ax2.grid(alpha=0.2); ax2.set_xlim(0, 1.02); ax2.set_ylim(0, 1.02)

fig.suptitle('XGBoost PU Bagging (Key Bands) — Performance on Test Set', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f'PR-AUC: {pr_auc:.4f}  |  ROC-AUC: {roc_auc:.4f}  |  N_test: {len(y_te)}')


## 5. 已知 CN 星标定阈值

用已知 CN 星概率的低分位标定候选阈值（保留约 95% 已知 CN 星），与基线做法一致。

In [ ]:
# 已知CN星标定候选阈值
from sklearn.neighbors import NearestNeighbors

known_mask = stars_clean['label'] == 1
known_scores = stars_clean.loc[known_mask, 'xgb_pu_prob'].values
recall_quantile = 0.05   # 保留约 95% 已知CN星

prob_threshold = float(np.nanquantile(known_scores, recall_quantile))

cand_mask = (stars_clean['label'] == -1) & (stars_clean['xgb_pu_prob'] >= prob_threshold)
candidates = stars_clean.loc[cand_mask].sort_values('xgb_pu_prob', ascending=False).copy()

n_known_above = int((known_scores >= prob_threshold).sum())

print(f"已知CN星概率: min={known_scores.min():.3f}  "
      f"q05={np.nanquantile(known_scores, 0.05):.3f}  "
      f"median={np.nanmedian(known_scores):.3f}  max={known_scores.max():.3f}")
print(f"标定阈值 (保留~{(1-recall_quantile)*100:.0f}%已知CN星): prob_threshold = {prob_threshold:.4f}")
print(f"已知CN星中 >= 阈值: {n_known_above}/{len(known_scores)}")
print(f"未标注候选体数量: {len(candidates)}  ({len(candidates)/len(stars_clean)*100:.2f}%)")

# 参数近邻平均光谱辅助函数（用于可视化对比，与基线一致）
_param_cols = ('teff', 'logg', 'feh')
n_param_nn = 50

def build_param_nn(stars, param_cols=_param_cols, n_neighbors=n_param_nn):
    P = stars[list(param_cols)].to_numpy(dtype=float)
    mu = np.nanmean(P, axis=0)
    sd = np.nanstd(P, axis=0)
    sd = np.where(sd == 0, 1.0, sd)
    Ps = (P - mu) / sd
    nn = NearestNeighbors(n_neighbors=min(n_neighbors + 1, len(P)), metric='euclidean')
    nn.fit(Ps)
    return nn, Ps

def param_cluster_mean_spec(idx, stars, Ps, X, n_neighbors=n_param_nn):
    cid_arr = stars['masked_cluster_id'].values
    cid = cid_arr[idx]
    cluster_pos = np.where(cid_arr == cid)[0]
    cluster_pos = cluster_pos[cluster_pos != idx]
    if len(cluster_pos) == 0:
        return None
    d = np.linalg.norm(Ps[cluster_pos] - Ps[idx], axis=1)
    order = np.argsort(d)
    sel = cluster_pos[order[:min(n_neighbors, len(cluster_pos))]]
    return np.nanmedian(X[sel], axis=0)

param_nn, Ps_all = build_param_nn(stars_clean, n_neighbors=n_param_nn)
print(f"参数近邻模型已构建 (n_neighbors={n_param_nn})")


## 6. 候选体可视化：光谱

展示候选体的**完整光谱**（归一化），并用色块标出作为模型输入的三个关键波段，便于直观判断 CN 增峰是否落在输入波段内。

In [ ]:
# 候选体光谱可视化（完整光谱 + 关键波段高亮）

top_n = 3
cand_vis = candidates.head(top_n)
cand_indices = cand_vis.index.values

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes = axes.flatten()

for i, idx in enumerate(cand_indices):
    ax = axes[i]
    flux = X_clean[idx]   # 完整 700 维光谱

    # 对比线：同一掩盖聚类内、三参数最接近的恒星平均光谱
    param_mean = param_cluster_mean_spec(idx, stars_clean, Ps_all, X_clean)
    ax.plot(common_wave, param_mean, color='seagreen',
            linewidth=1.0, linestyle='--', alpha=0.85,
            label='Cluster+Param-NN mean')
    ax.plot(common_wave, flux, color='navy', linewidth=0.8, label='Candidate')

    # 高亮作为模型输入的关键波段
    for l1, l2, c in [(3830, 3883, 'blue'), (4120, 4216, 'green'), (4285, 4315, 'red')]:
        ax.axvspan(l1, l2, alpha=0.12, color=c, zorder=0)

    prob = cand_vis.iloc[i]['xgb_pu_prob']
    teff = cand_vis.iloc[i]['teff']
    ax.set_title(f'#{i+1} | Teff={teff:.0f}K | prob={prob:.3f}', fontsize=8)
    ax.set_xlim(3800, 4500)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.2)
    ax.legend(fontsize=6, loc='upper right')

for j in range(top_n, len(axes)):
    axes[j].axis('off')

fig.suptitle('XGBoost PU Bagging (Key Bands) — Top Candidates Spectra (Input bands shaded)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## 7. 导出高置信度候选体（已知 CN 星标定阈值）

In [ ]:
# 导出结果
out_cols = ['uid', 'ra', 'dec', 'teff', 'logg', 'feh', 'label', 'snru',
            'xgb_pu_prob', 'xgb_pu_std', 'masked_cluster_id']
out_cols = [c for c in out_cols if c in candidates.columns]

out_df = candidates[out_cols].copy()
outpath = str(_PROJECT_ROOT / 'XGB/XGB_PU_keybands_candidates_threshold.csv')
out_df.to_csv(outpath, index=False)

print(f"候选体已导出: {outpath}")
print(f"筛选阈值 prob_threshold = {prob_threshold:.4f}  (保留~{(1-recall_quantile)*100:.0f}%已知CN星)")
print(f"候选体统计 (共 {len(out_df)} 颗):")
print(f"  prob > 0.5: {(out_df['xgb_pu_prob'] > 0.5).sum()}")
print(f"  prob > 0.7: {(out_df['xgb_pu_prob'] > 0.7).sum()}")


## 8. 结论（与全波段对比）

**关键波段（182 维） vs 全波段（700 维）对比要点：**

1. **判别信息集中度**：若关键波段的 ROC-AUC / PR-AUC 接近全波段，说明 CN 判别信息高度集中在三个分子带；若明显下降，说明连续谱背景（温度、金属丰度、光度）也贡献了不可忽略的信息
2. **过拟合与稳定性**：更低的输入维度通常带来更快的训练、更低的过拟合风险，但也可能丢失连续谱的物理上下文
3. **物理可解释性**：关键波段输入天然聚焦在 CN/CH 吸收特征上，候选体的可解释性更强
4. **与全波段候选体对比**：可进一步比较两份候选体 CSV 的重叠率，判断两者是否捕获同一批目标

> 将本 notebook 得到的 ROC-AUC / PR-AUC / P@50 / P@100 与基线 `ML_XGB_PU_threshold` 逐项对比即可得出结论。